In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

project = Path.cwd()
if not (project / "data").is_dir():
    project = project.parent

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from src.feature_preparation import (
    make_features,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    ENGINEERED_FEATURES,
)

dev = pd.read_csv(
    project / "data/processed/development.csv",
    dtype={"ID": "string"},
    low_memory=False
)

X_original = make_features(dev, engineered=False)
X_engineered = make_features(dev, engineered=True)
y = dev["Rej_and_cut_Piece"].copy()

assert set(dev["cv_fold"].unique()) == set(range(5))
assert dev.groupby("ID")["cv_fold"].nunique().eq(1).all()

print("Original feature table:", X_original.shape)
print("Engineered feature table:", X_engineered.shape)

print("\nMissing values after feature preparation:")
display(X_engineered.isna().sum())

display(X_engineered.head())

Original feature table: (16139, 8)
Engineered feature table: (16139, 11)

Missing values after feature preparation:


Req_Finish_Fabrics      0
Fabric_Allowance        0
Shrink_allow            0
Req_grey_fabric         0
Req_beam_length(yds)    0
epi                     0
ppi                     0
Construction            0
epi_ppi_product         0
epi_ppi_ratio           0
planned_grey_extra      0
dtype: int64

,Req_Finish_Fabrics,Fabric_Allowance,Shrink_allow,Req_grey_fabric,Req_beam_length(yds),epi,ppi,Construction,epi_ppi_product,epi_ppi_ratio,planned_grey_extra
0,31300,6.0,12.5,33297.872340,34797.651100,110,80,40+40/2/40/110x80,8800,1.375000,1997.872340
1,10450,7.0,14.5,11236.559140,12017.350430,110,90,40x40/110x90,9900,1.222222,786.559140
2,900,11.5,12.5,1016.949153,1062.753843,110,80,40x40/110x80,8800,1.375000,116.949153
3,8000,8.0,14.0,8695.652174,9245.815159,130,80,40x40/130x80,10400,1.625000,695.652174
4,3500,8.1,9.1,3808.487486,3831.158751,140,70,50x50/140x70,9800,2.000000,308.487486


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor(engineered=False, scale_numeric=False):
    numeric_columns = NUMERIC_FEATURES.copy()

    if engineered:
        numeric_columns += ENGINEERED_FEATURES

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ])


print("Preprocessing definitions are ready.")

Preprocessing definitions are ready.


In [3]:
# Use fold 0 as validation for this preprocessing check
training_rows = dev["cv_fold"].ne(0)
validation_rows = dev["cv_fold"].eq(0)

preprocessor_check = build_preprocessor(
    engineered=True,
    scale_numeric=True
)

training_array = preprocessor_check.fit_transform(
    X_engineered.loc[training_rows]
)

validation_array = preprocessor_check.transform(
    X_engineered.loc[validation_rows]
)

assert np.isfinite(training_array).all()
assert np.isfinite(validation_array).all()
assert training_array.shape[1] == validation_array.shape[1]

print("Training array:", training_array.shape)
print("Validation array:", validation_array.shape)
print("Preprocessing check passed.")

Training array: (12911, 104)
Validation array: (3228, 104)
Preprocessing check passed.


In [4]:
from time import perf_counter
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

models = {
    "Median baseline": DummyRegressor(strategy="median"),

    "Ridge": Ridge(alpha=1.0),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=150,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),

    "Histogram Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        early_stopping=False,
        random_state=42,
    ),
}

print("Median baseline and five regression models are ready.")

Median baseline and five regression models are ready.


In [5]:
feature_versions = {
    "Original": (X_original, False),
    "Engineered": (X_engineered, True),
}

fold_results = []
prediction_records = []
completed = 0

for feature_name, (X, engineered) in feature_versions.items():
    for model_name, estimator in models.items():

        # The median baseline does not depend on feature engineering
        if model_name == "Median baseline" and engineered:
            continue

        for fold in range(5):
            train_positions = np.flatnonzero(
                dev["cv_fold"].to_numpy() != fold
            )
            validation_positions = np.flatnonzero(
                dev["cv_fold"].to_numpy() == fold
            )

            # A fresh pipeline is fitted separately within every fold
            pipeline = Pipeline([
                (
                    "preprocessing",
                    build_preprocessor(
                        engineered=engineered,
                        scale_numeric=(model_name == "Ridge"),
                    ),
                ),
                ("model", clone(estimator)),
            ])

            start = perf_counter()
            pipeline.fit(
                X.iloc[train_positions],
                y.iloc[train_positions],
            )
            fit_seconds = perf_counter() - start

            start = perf_counter()
            predictions = pipeline.predict(
                X.iloc[validation_positions]
            )
            predict_seconds = perf_counter() - start

            actual = y.iloc[validation_positions]

            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(
                mean_squared_error(actual, predictions)
            )
            r2 = r2_score(actual, predictions)

            fold_results.append({
                "Feature_set": feature_name,
                "Model": model_name,
                "Fold": fold,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "Fit_seconds": fit_seconds,
                "Predict_seconds": predict_seconds,
            })

            prediction_records.append(pd.DataFrame({
                "source_row": dev.iloc[
                    validation_positions
                ]["source_row"].to_numpy(),
                "ID": dev.iloc[
                    validation_positions
                ]["ID"].to_numpy(),
                "Feature_set": feature_name,
                "Model": model_name,
                "Fold": fold,
                "Actual": actual.to_numpy(),
                "Predicted": predictions,
            }))

            completed += 1
            print(
                f"{completed}/55 | {feature_name} | "
                f"{model_name} | Fold {fold + 1} | "
                f"MAE: {mae:.2f}",
                flush=True,
            )

print("Initial grouped cross-validation completed.")

1/55 | Original | Median baseline | Fold 1 | MAE: 102.50
2/55 | Original | Median baseline | Fold 2 | MAE: 146.43
3/55 | Original | Median baseline | Fold 3 | MAE: 100.08
4/55 | Original | Median baseline | Fold 4 | MAE: 82.07
5/55 | Original | Median baseline | Fold 5 | MAE: 72.51
6/55 | Original | Ridge | Fold 1 | MAE: 137.70
7/55 | Original | Ridge | Fold 2 | MAE: 163.72
8/55 | Original | Ridge | Fold 3 | MAE: 143.12
9/55 | Original | Ridge | Fold 4 | MAE: 130.02
10/55 | Original | Ridge | Fold 5 | MAE: 110.75
11/55 | Original | Decision Tree | Fold 1 | MAE: 151.09
12/55 | Original | Decision Tree | Fold 2 | MAE: 157.34
13/55 | Original | Decision Tree | Fold 3 | MAE: 149.48
14/55 | Original | Decision Tree | Fold 4 | MAE: 142.57
15/55 | Original | Decision Tree | Fold 5 | MAE: 120.71
16/55 | Original | Random Forest | Fold 1 | MAE: 146.48
17/55 | Original | Random Forest | Fold 2 | MAE: 160.36
18/55 | Original | Random Forest | Fold 3 | MAE: 149.86
19/55 | Original | Random Forest 

In [6]:
results = pd.DataFrame(fold_results)

comparison = (
    results.groupby(["Feature_set", "Model"])
    .agg(
        MAE=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2=("R2", "mean"),
        R2_std=("R2", "std"),
        Fit_seconds=("Fit_seconds", "mean"),
        Predict_seconds=("Predict_seconds", "mean"),
    )
    .reset_index()
    .sort_values("MAE")
)

display(comparison.round(4))

results.to_csv(
    project / "reports/initial_cv_fold_results.csv",
    index=False,
)

comparison.to_csv(
    project / "reports/initial_model_comparison.csv",
    index=False,
)

pd.concat(prediction_records, ignore_index=True).to_csv(
    project / "data/processed/initial_cv_predictions.csv",
    index=False,
)

print("Comparison, fold results, and validation predictions saved.")

,Feature_set,Model,MAE,MAE_std,RMSE,RMSE_std,R2,R2_std,Fit_seconds,Predict_seconds
8,Original,Median baseline,100.7195,28.4464,459.2165,208.6317,-0.0442,0.0172,0.0443,0.0090
6,Original,Extra Trees,131.1547,27.4183,517.4753,205.0834,-0.4552,0.6710,4.0540,0.0777
1,Engineered,Extra Trees,131.4489,28.6036,520.8120,206.3293,-0.4863,0.7340,3.8134,0.0720
4,Engineered,Ridge,137.0136,21.4187,446.6047,208.1738,0.0219,0.0256,0.0543,0.0088
10,Original,Ridge,137.0645,19.2989,447.3002,206.9741,0.0171,0.0281,0.0501,0.0094
9,Original,Random Forest,137.8975,22.0729,547.1117,183.6057,-0.6740,0.6945,2.3136,0.0911
3,Engineered,Random Forest,138.9891,24.4694,548.1720,188.1862,-0.6769,0.7207,2.2139,0.0703
0,Engineered,Decision Tree,143.4796,17.0526,637.7945,134.8560,-1.7488,2.0817,0.1286,0.0090
5,Original,Decision Tree,144.2383,14.1638,643.1127,125.4965,-1.8092,2.0715,0.1372,0.0111
2,Engineered,Histogram Gradient Boosting,154.2724,19.9405,532.0487,197.4168,-0.5109,0.3792,1.5347,0.0327


Comparison, fold results, and validation predictions saved.
